# Compustat-CRSP Merge: Reprex Final — Extended (Investability Groupings)

Adds **investability groupings** (CRSP-style, using data we already collect) and **8 portfolios**: 2 (R&D) × 4 (investability).


## Setup: Install and load packages


In [1]:
# ============================================================================

import os
import sys
import pickle
import warnings
from datetime import datetime, date
from pathlib import Path

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine
import psycopg2
from scipy import stats
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

warnings.filterwarnings('ignore')

# Try to import optional packages
try:
    import wrds
    HAS_WRDS = True
except ImportError:
    HAS_WRDS = False
    print("Warning: wrds package not found. Install with: pip install wrds")


## Helper Functions


In [2]:
# ============================================================================

def connect_wharton(username="zkg232"):
    """Connect to WRDS PostgreSQL database."""
    if HAS_WRDS:
        # Using wrds package (recommended)
        # WRDS connection returns a connection object that has .raw_sql() method
        return wrds.Connection(wrds_username=username)
    else:
        # Fallback to direct PostgreSQL connection
        conn_string = (
            f"postgresql://{username}@wrds-pgdata.wharton.upenn.edu:9737/wrds"
            "?sslmode=require"
        )
        return create_engine(conn_string)

def execute_sql(query, con):
    """Execute SQL query and return DataFrame. Handles both WRDS and SQLAlchemy connections."""
    if HAS_WRDS and isinstance(con, wrds.Connection):
        # Use WRDS connection's raw_sql method (matches misc/assignment_1.ipynb pattern)
        return con.raw_sql(query)
    else:
        # Use pandas read_sql with SQLAlchemy engine
        return pd.read_sql(query, con)

def funda_name_mapping():
    """Return mapping of Compustat variable names to snake_case."""
    return pd.DataFrame({
        'variable_name': [
            'gvkey', 'datadate', 'fyear', 'fyr', 'apdedate', 'fdate', 'pdate',
            'conm', 'tic', 'cusip', 'cik', 'ni', 'xrd', 'gsector', 'permno',
            'date', 'ret_crsp', 'prc_crsp', 'vol', 'mktcap', 'ret_eff'
        ],
        'snake_name': [
            'gvkey', 'report_date', 'fiscal_year', 'fiscal_year_end_month',
            'actual_period_end_date', 'final_date', 'preliminary_date',
            'company_name', 'ticker', 'cusip', 'cik', 'net_income_loss',
            'research_and_development_expense', 'gic_sectors', 'permno',
            'date', 'ret_crsp', 'prc_crsp', 'vol', 'mktcap', 'ret_eff'
        ]
    })

def gics_mapping():
    """Return GICS sector code to name mapping."""
    return pd.DataFrame({
        'gic_sectors': ['10', '15', '20', '25', '30', '35', '40', '45', '50', '55', '60'],
        'sector_name': [
            'Energy', 'Materials', 'Industrials', 'Consumer Discretionary',
            'Consumer Staples', 'Health Care', 'Financials',
            'Information Technology', 'Communication Services', 'Utilities', 'Real Estate'
        ]
    })

def add_gics_names(data):
    """Add GICS sector names to dataframe."""
    gics_map = gics_mapping()
    if 'gic_sectors' in data.columns:
        data = data.merge(gics_map, on='gic_sectors', how='left')
    return data

def format_numeric_cols(data, rounding_map):
    """Format numeric columns with specified rounding."""
    data = data.copy()
    for col, digits in rounding_map.items():
        if col in data.columns and pd.api.types.is_numeric_dtype(data[col]):
            data[col] = data[col].round(digits)
    return data


## Data Loading and Merge


In [3]:
# ============================================================================

workspace_file = "notes/merged_data_workspace.pkl"

# Ensure notes directory exists
os.makedirs(os.path.dirname(workspace_file), exist_ok=True)

if os.path.exists(workspace_file):
    with open(workspace_file, 'rb') as f:
        workspace = pickle.load(f)
        merged = workspace.get('merged')
        con = workspace.get('con')
        start_date = workspace.get('start_date', pd.Timestamp('1980-01-01'))
        end_date = workspace.get('end_date', pd.Timestamp('2024-12-31'))
        start_year = workspace.get('start_year', 1980)
        end_year = workspace.get('end_year', 2024)
        cusip_target = workspace.get('cusip_target', None)
    
    # Reconnect if needed (WRDS connections can't be pickled, so always reconnect)
    con = connect_wharton()
else:
    con = connect_wharton()
    cusip_target = None
    start_year = 1980
    end_year = 2024
    start_date = pd.Timestamp('1980-01-01')
    end_date = pd.Timestamp('2024-12-31')
    
    # Build queries
    if cusip_target is not None:
        cusip_query = f"""
        SELECT DISTINCT gvkey
        FROM comp.names
        WHERE SUBSTR(cusip, 1, 8) = '{cusip_target}'
        """
        cusip_gvkey = execute_sql(cusip_query, con)
    else:
        cusip_gvkey = None
    
    # CCM link history
    ccm_query = """
    SELECT gvkey, lpermno as permno, lpermco as permco,
           linkdt, COALESCE(linkenddt, '9999-12-31'::date) as linkenddt
    FROM crsp.ccmxpf_lnkhist
    WHERE linktype IN ('LU', 'LC') AND linkprim IN ('P', 'C')
    """
    if cusip_gvkey is not None and len(cusip_gvkey) > 0:
        gvkeys = "', '".join(cusip_gvkey['gvkey'].astype(str))
        ccm_query += f" AND gvkey IN ('{gvkeys}')"
    
    ccm_link = execute_sql(ccm_query, con)
    ccm_link['linkdt'] = pd.to_datetime(ccm_link['linkdt'], errors='coerce')
    # Handle '9999-12-31' sentinel dates - convert to NaT or far future date
    ccm_link['linkenddt'] = pd.to_datetime(ccm_link['linkenddt'], errors='coerce')
    # Replace any overflow dates (like 9999-12-31) with a reasonable far future date
    max_date = pd.Timestamp('2099-12-31')
    ccm_link['linkenddt'] = ccm_link['linkenddt'].where(
        ccm_link['linkenddt'] <= max_date, 
        pd.NaT
    )
    
    # Compustat fundamentals
    funda_query = f"""
    SELECT f.gvkey, f.datadate, f.fyear, f.fyr, f.apdedate, f.fdate, f.pdate,
           f.conm, f.tic, f.cusip, f.cik, f.ni, f.xrd,
           SUBSTR(n.gind, 1, 2) as gsector,
           f.datadate as endfyr,
           f.datadate - INTERVAL '11 months' as begfyr,
           f.datadate + INTERVAL '3 months' as available_date
    FROM comp.funda f
    LEFT JOIN comp.names n ON f.gvkey = n.gvkey
    WHERE f.fyear >= {start_year - 10} AND f.fyear <= {end_year}
      AND f.indfmt = 'INDL' AND f.datafmt = 'STD' 
      AND f.popsrc = 'D' AND f.consol = 'C'
      AND f.curcd = 'USD' AND f.fic = 'USA'
      AND f.exchg BETWEEN 11 AND 19
      AND (f.sich IS NULL OR (f.sich NOT BETWEEN 6000 AND 6999 AND f.sich != 2834))
    """
    if cusip_gvkey is not None:
        gvkeys = "', '".join(cusip_gvkey['gvkey'].astype(str))
        funda_query += f" AND f.gvkey IN ('{gvkeys}')"
    
    funda = execute_sql(funda_query, con)
    funda['datadate'] = pd.to_datetime(funda['datadate'], errors='coerce')
    funda['endfyr'] = pd.to_datetime(funda['endfyr'], errors='coerce')
    funda['begfyr'] = pd.to_datetime(funda['begfyr'], errors='coerce')
    funda['available_date'] = pd.to_datetime(funda['available_date'], errors='coerce')
    
    # Drop rows with invalid dates
    funda = funda[
        funda['datadate'].notna() & 
        funda['endfyr'].notna() & 
        funda['begfyr'].notna() & 
        funda['available_date'].notna()
    ]
    
    # CRSP monthly stock file
    msf_query = f"""
    SELECT m.permno, m.date, m.ret, m.prc, m.vol, m.shrout,
           ABS(m.prc * m.shrout) as mktcap
    FROM crsp.msf m
    INNER JOIN crsp.msenames n ON m.permno = n.permno
    WHERE m.date >= '{start_date.date()}' AND m.date <= '{end_date.date()}'
      AND m.ret IS NOT NULL AND m.prc IS NOT NULL AND m.shrout IS NOT NULL
      AND m.date >= n.namedt AND m.date <= COALESCE(n.nameendt, '9999-12-31'::date)
      AND (n.siccd NOT BETWEEN 6000 AND 6999 AND n.siccd != 2834)
      AND n.shrcd IN (10, 11) AND n.exchcd IN (1, 2, 3)
    """
    msf = execute_sql(msf_query, con)
    msf['date'] = pd.to_datetime(msf['date'], errors='coerce')
    
    # Delisting returns
    delist_query = f"""
    SELECT permno, dlstdt as date, dlret
    FROM crsp.msedelist
    WHERE dlstdt >= '{start_date.date()}' AND dlstdt <= '{end_date.date()}'
      AND dlret IS NOT NULL
    """
    delist = execute_sql(delist_query, con)
    delist['date'] = pd.to_datetime(delist['date'], errors='coerce')
    delist['dlret'] = pd.to_numeric(delist['dlret'], errors='coerce')
    
    # Apply CCM linking
    link_msf = msf.merge(ccm_link, on='permno', how='inner')
    # Handle NaT in linkenddt (represents open-ended links)
    link_msf = link_msf[
        (link_msf['date'] >= link_msf['linkdt']) &
        (link_msf['linkenddt'].isna() | (link_msf['date'] <= link_msf['linkenddt']))
    ]
    link_msf = link_msf.sort_values(['permno', 'date', 'linkdt'])
    link_msf = link_msf.drop_duplicates(subset=['permno', 'date'], keep='last')
    
    # Merge logic
    crsp_dates = link_msf[['gvkey', 'permno', 'date']].drop_duplicates()
    
    funda_filtered = funda[funda['available_date'] <= end_date].copy()
    
    merged = crsp_dates.merge(
        funda_filtered, on='gvkey', how='inner', suffixes=('', '_funda')
    )
    merged = merged[merged['date'] >= merged['available_date']]
    merged = merged.sort_values(['gvkey', 'permno', 'date', 'datadate'])
    merged = merged.drop_duplicates(subset=['gvkey', 'permno', 'date'], keep='last')
    
    # Join CRSP data
    crsp_data = link_msf[['gvkey', 'permno', 'date', 'ret', 'prc', 'vol', 'mktcap']].copy()
    crsp_data = crsp_data.rename(columns={'ret': 'ret_crsp', 'prc': 'prc_crsp'})
    crsp_data['prc_crsp'] = crsp_data['prc_crsp'].abs()
    crsp_data['ret_crsp'] = pd.to_numeric(crsp_data['ret_crsp'], errors='coerce')
    
    merged = merged.merge(crsp_data, on=['gvkey', 'permno', 'date'], how='inner')
    merged = merged.merge(delist, on=['permno', 'date'], how='left')
    
    # Calculate effective returns
    merged['ret_eff'] = (
        (1 + merged['ret_crsp'].fillna(0)) * 
        (1 + merged['dlret'].fillna(0)) - 1
    )
    
    # Clean up
    merged = merged.drop(columns=['begfyr', 'endfyr', 'available_date', 'dlret'], errors='ignore')
    merged = merged[
        ~merged['ret_crsp'].astype(str).str.contains('[A-Za-z]', na=False) &
        (pd.to_numeric(merged['ret_crsp'], errors='coerce') >= -100)
    ]
    
    # Rename columns
    name_map = funda_name_mapping()
    rename_dict = dict(zip(name_map['variable_name'], name_map['snake_name']))
    merged = merged.rename(columns=rename_dict)
    
    # Add GICS names
    merged = add_gics_names(merged)
    
    # Save workspace (don't save connection - WRDS connections can't be pickled)
    workspace = {
        'merged': merged, 'start_year': start_year, 'end_year': end_year,
        'start_date': start_date, 'end_date': end_date, 'cusip_target': cusip_target
    }
    with open(workspace_file, 'wb') as f:
        pickle.dump(workspace, f)

# Ensure connection (WRDS connections can't be pickled, so always reconnect if needed)
if 'con' not in locals() or con is None:
    con = connect_wharton()
if 'start_date' not in locals():
    start_date = pd.Timestamp('1980-01-01')
    end_date = pd.Timestamp('2024-12-31')
    start_year = 1980
    end_year = 2024
    cusip_target = None


WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


## Statistical Analysis Functions


In [4]:
# ============================================================================

def calculate_portfolio_returns(data, group_var='research_not'):
    """Calculate equal-weighted and value-weighted portfolio returns."""
    data = data[
        data['ret_crsp'].notna() & data['prc_crsp'].notna() & 
        data['mktcap'].notna() & (data['ret_crsp'] != 0)
    ].copy()
    
    data = data.sort_values(['permno', 'date'])
    data['mktcap_lag'] = data.groupby('permno')['mktcap'].shift(1)
    data['prc_lag'] = data.groupby('permno')['prc_crsp'].shift(1)
    data = data[data['mktcap_lag'].notna() & data['prc_lag'].notna()]
    
    # Ceiling to month end
    data['date'] = data['date'] + pd.offsets.MonthEnd(0)
    
    # Calculate VW return
    def calc_vw_return(group):
        total_mktcap = group['mktcap_lag'].sum()
        if total_mktcap == 0:
            return np.nan
        weights = group['mktcap_lag'] / total_mktcap
        return (weights * group['ret_crsp']).sum()
    
    portfolio_returns = data.groupby(['date', group_var]).apply(
        lambda x: pd.Series({
            'vw_return': calc_vw_return(x),
            'ew_return': x['ret_crsp'].mean(),
            'total_mktcap': x['mktcap_lag'].sum(),
            'n_stocks': len(x)
        })
    ).reset_index()
    
    return portfolio_returns.sort_values([group_var, 'date'])

def calculate_market_return(con, start_date, end_date):
    """Get CRSP market index returns."""
    query = f"""
    SELECT date, vwretd, ewretd
    FROM crsp.msi
    WHERE date >= '{start_date.date()}' AND date <= '{end_date.date()}'
      AND vwretd IS NOT NULL AND ewretd IS NOT NULL
    ORDER BY date
    """
    market_returns = execute_sql(query, con)
    market_returns['date'] = pd.to_datetime(market_returns['date'], errors='coerce')
    market_returns['vwretd'] = pd.to_numeric(market_returns['vwretd'], errors='coerce')
    market_returns['ewretd'] = pd.to_numeric(market_returns['ewretd'], errors='coerce')
    return market_returns

def run_capm_regression(portfolio_returns, market_returns, rf_rate=0, use_ew_benchmark=False):
    """Run CAPM regression and return metrics."""
    market_col = 'ewretd' if use_ew_benchmark else 'vwretd'
    
    # Merge and calculate excess returns
    portfolio_returns = portfolio_returns.copy()
    portfolio_returns['date'] = portfolio_returns['date'] + pd.offsets.MonthEnd(0)
    
    market_data = market_returns[['date', market_col]].copy()
    market_data['date'] = market_data['date'] + pd.offsets.MonthEnd(0)
    market_data = market_data.rename(columns={market_col: 'market_return'})
    
    reg_data = portfolio_returns.merge(market_data, on='date', how='left')
    reg_data = reg_data[
        reg_data['portfolio_return'].notna() & reg_data['market_return'].notna()
    ].copy()
    
    reg_data['portfolio_excess'] = reg_data['portfolio_return'] - rf_rate
    reg_data['market_excess'] = reg_data['market_return'] - rf_rate
    
    if len(reg_data) < 2:
        return None
    
    # Ensure numeric types and drop any remaining NaN
    reg_data = reg_data[
        reg_data['portfolio_excess'].notna() & 
        reg_data['market_excess'].notna()
    ].copy()
    
    if len(reg_data) < 2:
        return None
    
    # Convert to numeric arrays explicitly
    market_excess = pd.to_numeric(reg_data['market_excess'], errors='coerce')
    portfolio_excess = pd.to_numeric(reg_data['portfolio_excess'], errors='coerce')
    
    # Drop any NaN that resulted from conversion
    valid_mask = market_excess.notna() & portfolio_excess.notna()
    market_excess = market_excess[valid_mask]
    portfolio_excess = portfolio_excess[valid_mask]
    
    if len(market_excess) < 2:
        return None
    
    # Run regression (force float numpy arrays; avoids object dtype from pandas nullable arrays)
    market_excess_arr = market_excess.to_numpy(dtype=float)
    portfolio_excess_arr = portfolio_excess.to_numpy(dtype=float)

    X = pd.DataFrame({'market_excess': market_excess_arr})
    X = add_constant(X, has_constant='add')
    y = portfolio_excess_arr
    model = OLS(y, X).fit()
    
    n_months = len(portfolio_excess_arr)
    # Get original portfolio return for arithmetic mean
    portfolio_return_vals = pd.to_numeric(reg_data.loc[valid_mask, 'portfolio_return'], errors='coerce')
    arithmetic_monthly_return = portfolio_return_vals.to_numpy(dtype=float).mean()
    monthly_vol = portfolio_excess_arr.std(ddof=1)
    
    # Coerce regression outputs to labeled pandas objects (robust to ndarray outputs)
    def _as_series(x, index):
        if isinstance(x, pd.Series):
            return x
        arr = np.asarray(x).reshape(-1)
        return pd.Series(arr, index=list(index)[: len(arr)])

    params = _as_series(model.params, X.columns)
    tvalues = _as_series(model.tvalues, X.columns)
    pvalues = _as_series(model.pvalues, X.columns)

    ci = model.conf_int(alpha=0.05)
    if isinstance(ci, pd.DataFrame):
        conf_int = ci
    else:
        conf_int = pd.DataFrame(np.asarray(ci), index=params.index, columns=['lower', 'upper'])

    alpha = params.get('const', params.iloc[0] if len(params) > 0 else np.nan)
    beta = params.get('market_excess', params.iloc[1] if len(params) > 1 else np.nan)
    alpha_tstat = tvalues.get('const', tvalues.iloc[0] if len(tvalues) > 0 else np.nan)
    beta_tstat = tvalues.get('market_excess', tvalues.iloc[1] if len(tvalues) > 1 else np.nan)
    alpha_pval = pvalues.get('const', pvalues.iloc[0] if len(pvalues) > 0 else np.nan)
    beta_pval = pvalues.get('market_excess', pvalues.iloc[1] if len(pvalues) > 1 else np.nan)

    alpha_lower_ci = conf_int.iloc[0, 0] if len(conf_int) > 0 else np.nan
    alpha_upper_ci = conf_int.iloc[0, 1] if len(conf_int) > 0 else np.nan
    beta_lower_ci = conf_int.iloc[1, 0] if len(conf_int) > 1 else np.nan
    beta_upper_ci = conf_int.iloc[1, 1] if len(conf_int) > 1 else np.nan
    
    return pd.Series({
        'arithmetic_monthly_return': arithmetic_monthly_return,
        'annualized_return': (1 + arithmetic_monthly_return) ** 12 - 1,
        'alpha': alpha,
        'beta': beta,
        'alpha_tstat': alpha_tstat,
        'beta_tstat': beta_tstat,
        'alpha_pval': alpha_pval,
        'beta_pval': beta_pval,
        'alpha_lower_ci': alpha_lower_ci,
        'alpha_upper_ci': alpha_upper_ci,
        'beta_lower_ci': beta_lower_ci,
        'beta_upper_ci': beta_upper_ci,
        'r_squared': model.rsquared,
        'sharpe_ratio': (arithmetic_monthly_return - rf_rate) / monthly_vol * np.sqrt(12) if monthly_vol > 0 else np.nan,
        'annualized_alpha': alpha * 12,
        'annualized_vol': monthly_vol * np.sqrt(12),
        'n_observations': n_months
    })

def calculate_all_portfolio_metrics(portfolio_data, market_data, portfolio_name, rf_rate=0):
    """Calculate metrics for all portfolios (EW and VW)."""
    group_col = portfolio_data.columns[1]  # Second column after date
    
    # EW metrics
    ew_returns = portfolio_data[['date', 'ew_return', group_col]].copy()
    ew_returns = ew_returns.rename(columns={'ew_return': 'portfolio_return'})
    
    def run_regression_safe(group):
        result = run_capm_regression(
            group[['date', 'portfolio_return']], market_data, rf_rate, use_ew_benchmark=True
        )
        if result is None:
            return pd.Series(dtype=float)
        return result
    
    ew_metrics_list = []
    for group_name, group_data in ew_returns.groupby(group_col):
        result = run_regression_safe(group_data)
        if result is not None and len(result) > 0:
            result_dict = result.to_dict()
            result_dict[group_col] = group_name
            ew_metrics_list.append(result_dict)
    
    if ew_metrics_list:
        ew_metrics = pd.DataFrame(ew_metrics_list)
        ew_metrics['weighting'] = 'Equal-Weighted'
    else:
        ew_metrics = pd.DataFrame()
    
    # VW metrics
    vw_returns = portfolio_data[['date', 'vw_return', group_col]].copy()
    vw_returns = vw_returns.rename(columns={'vw_return': 'portfolio_return'})
    
    def run_regression_safe_vw(group):
        result = run_capm_regression(
            group[['date', 'portfolio_return']], market_data, rf_rate, use_ew_benchmark=False
        )
        if result is None:
            return pd.Series(dtype=float)
        return result
    
    vw_metrics_list = []
    for group_name, group_data in vw_returns.groupby(group_col):
        result = run_regression_safe_vw(group_data)
        if result is not None and len(result) > 0:
            result_dict = result.to_dict()
            result_dict[group_col] = group_name
            vw_metrics_list.append(result_dict)
    
    if vw_metrics_list:
        vw_metrics = pd.DataFrame(vw_metrics_list)
        vw_metrics['weighting'] = 'Value-Weighted'
    else:
        vw_metrics = pd.DataFrame()
    
    # Combine
    if len(ew_metrics) > 0 and len(vw_metrics) > 0:
        all_metrics = pd.concat([ew_metrics, vw_metrics], ignore_index=True)
        all_metrics['portfolio'] = all_metrics['weighting'] + '_' + all_metrics[group_col].astype(str)
        all_metrics = all_metrics.drop(columns=[group_col])
    elif len(ew_metrics) > 0:
        all_metrics = ew_metrics.copy()
        all_metrics['portfolio'] = all_metrics['weighting'] + '_' + all_metrics[group_col].astype(str)
        all_metrics = all_metrics.drop(columns=[group_col])
    elif len(vw_metrics) > 0:
        all_metrics = vw_metrics.copy()
        all_metrics['portfolio'] = all_metrics['weighting'] + '_' + all_metrics[group_col].astype(str)
        all_metrics = all_metrics.drop(columns=[group_col])
    else:
        all_metrics = pd.DataFrame()
    
    return all_metrics

def calculate_sharpe_ratio(returns, rf_rate=0):
    """Calculate annualized Sharpe ratio."""
    if len(returns) == 0 or returns.isna().all():
        return np.nan
    excess_returns = returns - rf_rate
    mean_excess = excess_returns.mean()
    sd_returns = returns.std()
    if sd_returns == 0 or pd.isna(sd_returns):
        return np.nan
    return (mean_excess / sd_returns) * np.sqrt(12)


## Data Preparation


### Investability (yes/no — data we already collect)

We only use **existing** merged fields (no new WRDS pulls). **Pass** = Yes, **Fail** = No.

- **Size (CRSP add threshold):** `mktcap_lag >= $15 million`.
- **Has prior 12 prices:** at least 12 months of valid `prc_crsp` in the 12 months before the current month (seasoning/liquidity proxy).

**Investability = Yes** iff both checks pass; else **No**. That gives **2 (R&D) × 2 (investability) = 4 portfolio groups**, reported as **8** rows (EW and VW for each).


In [5]:
# ============================================================================

analysis_data = merged[[
    'gvkey', 'permno', 'date', 'report_date', 'company_name', 'ticker',
    'sector_name', 'ret_crsp', 'mktcap', 'net_income_loss',
    'research_and_development_expense', 'prc_crsp'
]].copy()
analysis_data = analysis_data.rename(columns={'sector_name': 'sector'})
analysis_data['research_not'] = analysis_data['research_and_development_expense'].apply(
    lambda x: 'Without_XRD' if pd.isna(x) else 'With_XRD'
)

# Investability: yes/no flag (size >= $15M and has prior 12 prices)
analysis_data = analysis_data.sort_values(['permno', 'date'])
analysis_data['mktcap_lag'] = analysis_data.groupby('permno')['mktcap'].shift(1)

# Prior 12 months: count months with valid price (point-in-time)
prior_12_prc_count = (
    analysis_data.groupby('permno')['prc_crsp']
    .transform(lambda x: x.shift(1).rolling(12, min_periods=12).count())
)
analysis_data['has_prior_12_prices'] = (prior_12_prc_count >= 12)

# Pass = Yes iff size >= $15M and has prior 12 prices; else No
size_ok = (analysis_data['mktcap_lag'] >= 15e6) & analysis_data['mktcap_lag'].notna()
analysis_data['investability'] = np.where(
    size_ok & analysis_data['has_prior_12_prices'], 'Yes', 'No'
)

analysis_data['portfolio_id'] = analysis_data['research_not'] + '_' + analysis_data['investability']

market_returns = calculate_market_return(con, start_date, end_date)
portfolio_returns = calculate_portfolio_returns(analysis_data, 'research_not')
portfolio_returns_8 = calculate_portfolio_returns(analysis_data, 'portfolio_id')
portfolio_returns_sector = analysis_data[
    analysis_data['sector'].notna()
].pipe(calculate_portfolio_returns, group_var='sector')

rf_rate_annual = 0.03
rf_rate_monthly = rf_rate_annual / 12

portfolio_metrics = calculate_all_portfolio_metrics(
    portfolio_returns, market_returns, 'R&D Portfolio', rf_rate_monthly
)
portfolio_metrics_8 = calculate_all_portfolio_metrics(
    portfolio_returns_8, market_returns, 'R&D × Investability', rf_rate_monthly
)

# Sector metrics
sector_metrics_list = []
for sector in portfolio_returns_sector['sector'].unique():
    sector_data = portfolio_returns_sector[
        portfolio_returns_sector['sector'] == sector
    ][['date', 'vw_return']].copy()
    sector_data = sector_data.rename(columns={'vw_return': 'portfolio_return'})
    metrics = run_capm_regression(
        sector_data, market_returns, rf_rate_monthly, use_ew_benchmark=False
    )
    if metrics is not None and len(metrics) > 0:
        metrics_dict = metrics.to_dict()
        metrics_dict['sector'] = sector
        sector_metrics_list.append(metrics_dict)

if sector_metrics_list:
    sector_metrics = pd.DataFrame(sector_metrics_list)
    sector_metrics = sector_metrics.sort_values('annualized_return', ascending=False)
else:
    sector_metrics = pd.DataFrame()


## Table Formatting Functions


In [6]:
# ============================================================================

def format_performance_table(metrics):
    """Format portfolio performance metrics table."""
    rounding_map = {
        'arithmetic_monthly_return': 6, 'annualized_return': 4, 'alpha': 6, 'beta': 6,
        'alpha_tstat': 6, 'beta_tstat': 2, 'alpha_lower_ci': 6, 'alpha_upper_ci': 6,
        'beta_lower_ci': 6, 'beta_upper_ci': 6, 'r_squared': 6, 'sharpe_ratio': 6
    }
    
    table = metrics.copy()
    table['Portfolio'] = table['portfolio']
    table = format_numeric_cols(table, rounding_map)
    table['Alpha p-value'] = table['alpha_pval'].apply(lambda x: f"{x:.3e}")
    table['Beta p-value'] = table['beta_pval'].apply(lambda x: f"{x:.3e}")
    
    return table[[
        'Portfolio', 'arithmetic_monthly_return', 'annualized_return', 'alpha',
        'beta', 'alpha_tstat', 'beta_tstat', 'Alpha p-value', 'Beta p-value',
        'alpha_lower_ci', 'alpha_upper_ci', 'beta_lower_ci', 'beta_upper_ci',
        'r_squared', 'sharpe_ratio'
    ]].rename(columns={
        'arithmetic_monthly_return': 'Arithmetic Monthly Return',
        'annualized_return': 'Annualized Return',
        'alpha': 'Alpha (Monthly)',
        'beta': 'Beta',
        'alpha_tstat': 'Alpha t-stat',
        'beta_tstat': 'Beta t-stat',
        'alpha_lower_ci': 'Alpha Lower CI (95%)',
        'alpha_upper_ci': 'Alpha Upper CI (95%)',
        'beta_lower_ci': 'Beta Lower CI (95%)',
        'beta_upper_ci': 'Beta Upper CI (95%)',
        'r_squared': 'R-squared',
        'sharpe_ratio': 'Sharpe Ratio'
    })

def format_sector_table(metrics):
    """Format sector performance metrics table."""
    rounding_map = {
        'annualized_return': 4, 'annualized_alpha': 4, 'beta': 6,
        'sharpe_ratio': 4, 'r_squared': 4
    }
    
    table = metrics.copy()
    table['Sector'] = table['sector']
    table = format_numeric_cols(table, rounding_map)
    
    return table[[
        'Sector', 'annualized_return', 'annualized_alpha', 'beta',
        'sharpe_ratio', 'r_squared', 'n_observations'
    ]].rename(columns={
        'annualized_return': 'Annualized Return',
        'annualized_alpha': 'Alpha (Annualized)',
        'beta': 'Beta',
        'sharpe_ratio': 'Sharpe Ratio',
        'r_squared': 'R-squared',
        'n_observations': 'N Observations'
    })


## Chart Functions


In [ ]:
# ============================================================================

def create_performance_chart(chart_data, panel_type='vw', recessions=None, colors=None, start_date=None):
    """Create performance chart for VW or EW portfolios; benchmark is always CRSP VW."""
    pattern = 'vw_return' if panel_type == 'vw' else 'ew_return'
    benchmark_name = 'CRSP VW Benchmark'
    panel_label = 'Panel A: Value-Weighted Portfolios' if panel_type == 'vw' else 'Panel B: Equal-Weighted Portfolios'
    
    panel_data = chart_data[chart_data['key'].str.contains(pattern)].copy()
    if panel_type == 'ew':
        vw_bench = chart_data[chart_data['key'] == 'vw_return_CRSP_VW_Benchmark'].copy()
        if len(vw_bench) > 0:
            panel_data = pd.concat([panel_data, vw_bench], ignore_index=True)
    panel_data['portfolio_name'] = panel_data['key'].apply(
        lambda x: 'R&D Portfolio' if 'With_XRD' in x
        else 'No R&D Portfolio' if 'Without_XRD' in x
        else benchmark_name if 'CRSP' in x else x
    )
    
    fig = go.Figure()
    
    for portfolio in panel_data['portfolio_name'].unique():
        port_data = panel_data[panel_data['portfolio_name'] == portfolio].sort_values('date')
        color = colors.get(portfolio, '#000000')
        fig.add_trace(go.Scatter(
            x=port_data['date'],
            y=port_data['cumvalue'],
            mode='lines',
            name=portfolio,
            line=dict(color=color, width=5),
            hovertemplate=f'<b>{portfolio}</b><br>Date: %{{x|%Y-%m}}<br>Value: $%{{y:.2f}}<extra></extra>'
        ))
    
    # Add recession shading
    if recessions is not None and len(recessions) > 0:
        for _, rec in recessions.iterrows():
            fig.add_vrect(
                x0=rec['start'], x1=rec['end'],
                fillcolor='rgba(128, 128, 128, 0.2)',
                layer='below',
                line_width=0
            )
    
    fig.update_layout(
        title={
            'text': 'Cumulative Performance Comparison (1980-2022)',
            'font': {'size': 25, 'color': 'white'}
        },
        xaxis_title='Date',
        yaxis_title='Cumulative Value ($)',
        yaxis=dict(rangemode='tozero', tickformat='$.2f'),
        template='plotly_dark',
        hovermode='x unified',
        legend=dict(orientation='v', yanchor='top', xanchor='right', x=1, y=1),
        height=600
    )
    
    return fig


## Alpha Table Generation


In [ ]:
# ============================================================================

def create_alpha_table(portfolio_metrics, market_returns, rf_rate_monthly):
    """Create alpha estimation results table."""
    benchmark_sharpe_vw = calculate_sharpe_ratio(market_returns['vwretd'], rf_rate_monthly)
    
    benchmark_rows = pd.DataFrame({
        'Portfolio': ['CRSP VW Benchmark'],
        'Alpha (monthly %)': ['—'],
        't-statistic': ['—'],
        'p-value': ['—'],
        'Beta': ['—'],
        'R²': ['—'],
        'Sharpe': [round(benchmark_sharpe_vw, 2)],
        'is_significant': [False],
        'sort_order': [99]
    })
    
    alpha_table = portfolio_metrics.copy()
    def portfolio_display_name(x):
        if 'Value-Weighted' in x and 'With_XRD' in x and '_Yes' in x: return 'R&D Investable VW'
        if 'Value-Weighted' in x and 'With_XRD' in x and '_No' in x: return 'R&D Not invest. VW'
        if 'Value-Weighted' in x and 'Without_XRD' in x and '_Yes' in x: return 'No R&D Investable VW'
        if 'Value-Weighted' in x and 'Without_XRD' in x and '_No' in x: return 'No R&D Not invest. VW'
        if 'Equal-Weighted' in x and 'With_XRD' in x and '_Yes' in x: return 'R&D Investable EW'
        if 'Equal-Weighted' in x and 'With_XRD' in x and '_No' in x: return 'R&D Not invest. EW'
        if 'Equal-Weighted' in x and 'Without_XRD' in x and '_Yes' in x: return 'No R&D Investable EW'
        if 'Equal-Weighted' in x and 'Without_XRD' in x and '_No' in x: return 'No R&D Not invest. EW'
        if 'Equal-Weighted' in x and 'With_XRD' in x: return 'R&D EW'
        if 'Equal-Weighted' in x and 'Without_XRD' in x: return 'No R&D EW'
        if 'Value-Weighted' in x and 'With_XRD' in x: return 'R&D VW'
        if 'Value-Weighted' in x and 'Without_XRD' in x: return 'No R&D VW'
        return x
    alpha_table['Portfolio'] = alpha_table['portfolio'].apply(portfolio_display_name)
    
    alpha_table['alpha_monthly_pct'] = (alpha_table['alpha'] * 100).round(2)
    alpha_table['alpha_star'] = alpha_table['alpha_pval'].apply(
        lambda x: '***' if x < 0.01 else '**' if x < 0.05 else '*' if x < 0.10 else ''
    )
    alpha_table['Alpha (monthly %)'] = alpha_table['alpha_monthly_pct'].apply(
        lambda x: f"{x:.2f}"
    ) + alpha_table['alpha_star']
    alpha_table['t-statistic'] = alpha_table['alpha_tstat'].round(2)
    alpha_table['p-value'] = alpha_table['alpha_pval'].apply(lambda x: f"{x:.3f}")
    alpha_table['Beta'] = alpha_table['beta'].round(2)
    alpha_table['R²'] = alpha_table['r_squared'].round(2)
    alpha_table['Sharpe'] = alpha_table['sharpe_ratio'].round(2)
    alpha_table['is_significant'] = alpha_table['alpha_pval'] < 0.10
    sort_map = {
        'R&D VW': 1, 'No R&D VW': 2, 'R&D EW': 3, 'No R&D EW': 4,
        'R&D Investable VW': 1, 'R&D Not invest. VW': 2, 'No R&D Investable VW': 3, 'No R&D Not invest. VW': 4,
        'R&D Investable EW': 5, 'R&D Not invest. EW': 6, 'No R&D Investable EW': 7, 'No R&D Not invest. EW': 8,
    }
    alpha_table['sort_order'] = alpha_table['Portfolio'].map(sort_map).fillna(9)
    
    alpha_table = alpha_table.sort_values('sort_order')
    alpha_table = alpha_table[[
        'Portfolio', 'Alpha (monthly %)', 't-statistic', 'p-value',
        'Beta', 'R²', 'Sharpe', 'is_significant', 'sort_order'
    ]].astype(str)
    
    benchmark_rows = benchmark_rows.astype(str)
    alpha_table = pd.concat([alpha_table, benchmark_rows], ignore_index=True)
    alpha_table = alpha_table.sort_values('sort_order').drop(columns='sort_order')
    
    return alpha_table

def create_alpha_table_html(alpha_table, custom_css):
    """Create HTML table with custom styling."""
    alpha_table_display = alpha_table.drop(columns='is_significant', errors='ignore')
    
    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
    {custom_css}
    </head>
    <body>
    <table class="table table-dark table-striped">
    <caption><b>Alpha Estimation Results: Single-Factor Model</b><br>
    <i>Rp - Rƒ = α + β(R_CRSP_VW - Rƒ)</i></caption>
    <thead>
    <tr>
    <th>Portfolio</th>
    <th>Alpha (monthly %)</th>
    <th>t-statistic</th>
    <th>p-value</th>
    <th>Beta</th>
    <th>R²</th>
    <th>Sharpe</th>
    </tr>
    </thead>
    <tbody>
    """
    
    for _, row in alpha_table_display.iterrows():
        portfolio = row['Portfolio']
        color = '#dd5182' if portfolio in ['R&D VW', 'R&D EW'] else \
                '#ffa600' if portfolio in ['No R&D VW', 'No R&D EW'] else 'white'
        bold = 'font-weight: bold;' if portfolio in ['R&D VW', 'R&D EW', 'No R&D VW', 'No R&D EW', 
                                                      'CRSP VW Benchmark'] else ''
        html += f"""
        <tr style="color: {color}; {bold}">
        <td>{portfolio}</td>
        <td>{row['Alpha (monthly %)']}</td>
        <td>{row['t-statistic']}</td>
        <td>{row['p-value']}</td>
        <td>{row['Beta']}</td>
        <td>{row['R²']}</td>
        <td>{row['Sharpe']}</td>
        </tr>
        """
    
    html += """
    </tbody>
    </table>
    </body>
    </html>
    """
    
    return html


## Main Execution


In [ ]:
# ============================================================================

performance_table = format_performance_table(portfolio_metrics)
performance_table_8 = format_performance_table(portfolio_metrics_8)
sector_table = format_sector_table(sector_metrics)

# Prepare chart data (2 portfolios + 4 investability portfolios)
chart_data = portfolio_returns.melt(
    id_vars=['date', 'research_not'],
    value_vars=['ew_return', 'vw_return'],
    var_name='weighting',
    value_name='value'
)
chart_data['key'] = chart_data['weighting'] + '_' + chart_data['research_not']

chart_data_8 = portfolio_returns_8.melt(
    id_vars=['date', 'portfolio_id'],
    value_vars=['ew_return', 'vw_return'],
    var_name='weighting',
    value_name='value'
)
chart_data_8['key'] = chart_data_8['weighting'] + '_' + chart_data_8['portfolio_id']
chart_data = pd.concat([chart_data, chart_data_8[['date', 'key', 'value']]], ignore_index=True)

# Add benchmarks
market_vw = market_returns.copy()
market_vw['date'] = market_vw['date'] + pd.offsets.MonthEnd(0)
market_vw['key'] = 'vw_return_CRSP_VW_Benchmark'
market_vw['value'] = market_vw['vwretd']
chart_data = pd.concat([
    chart_data,
    market_vw[['date', 'key', 'value']]
], ignore_index=True)


chart_data = chart_data[chart_data['date'] >= start_date].sort_values(['key', 'date'])

# Use transform/cumprod so the result aligns to the original index
chart_data['cumvalue'] = (1.0 + pd.to_numeric(chart_data['value'], errors='coerce')).groupby(chart_data['key']).cumprod()

recessions = pd.DataFrame({
    'start': pd.to_datetime(['1980-01-01', '1981-07-01', '1990-07-01', '2001-03-01', '2007-12-01', '2020-02-01']),
    'end': pd.to_datetime(['1980-07-01', '1982-11-01', '1991-03-01', '2001-11-01', '2009-06-01', '2020-04-01'])
})
recessions = recessions[
    (recessions['start'] >= start_date) & (recessions['start'] <= end_date)
]

colors = {
    'R&D Portfolio': '#dd5182',
    'No R&D Portfolio': '#ffa600',
    'R&D, Investable': '#dd5182',
    'R&D, Not investable': '#955196',
    'No R&D, Investable': '#ffa600',
    'No R&D, Not investable': '#003f5c',
    'CRSP VW Benchmark': '#58508d'
}

chart_a = create_performance_chart(chart_data, 'vw', recessions, colors, start_date)
chart_b = create_performance_chart(chart_data, 'ew', recessions, colors, start_date)

custom_css = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Tomorrow:wght@400;600;700&display=swap');
body { font-family: 'Tomorrow', sans-serif; background-color: #1d1d1c; padding: 1rem; margin: 0; color: white; }
:root { font-size: 25px; }
.table-dark.table-striped tbody tr:nth-of-type(odd) { background-color: rgba(255, 255, 255, 0.05) !important; }
.table-dark.table-striped tbody tr:nth-of-type(even) { background-color: rgba(0, 0, 0, 0.15) !important; }
.table-dark tbody tr[style*='color'] { background-color: inherit !important; }
.table-dark td, .table-dark th { padding: 0.5rem 0.75rem !important; }
</style>
"""

alpha_table = create_alpha_table(portfolio_metrics, market_returns, rf_rate_monthly)
alpha_table_8 = create_alpha_table(portfolio_metrics_8, market_returns, rf_rate_monthly)
alpha_table_html_wrapped = create_alpha_table_html(alpha_table, custom_css)
alpha_table_8_html_wrapped = create_alpha_table_html(alpha_table_8, custom_css)

# Display outputs (same format as Nick/our_outputs for 3 vs 4 month lag comparison)
if __name__ == '__main__' or 'get_ipython' in globals():
    # Display tables
    print("\n=== Reporting lag: 3 months (reprex_final_extended) ===")
    print("\nPerformance Table (2 portfolios):")
    print(performance_table.to_string())
    
    print("\nPerformance Table — 8 portfolios (R&D × Investability):")
    print(performance_table_8.to_string())
    
    print("\nSector Table:")
    print(sector_table.to_string())
    
    # Display charts
    chart_a.show()
    chart_b.show()
    
    # Display alpha table HTML (2 portfolios and 8 portfolios)
    from IPython.display import HTML, display
    display(HTML("<h3>Alpha table (2 portfolios) — 3-month lag</h3>"))
    display(HTML(alpha_table_html_wrapped))
    display(HTML("<h3>Alpha table — 8 portfolios (R&D × Investability) — 3-month lag</h3>"))
    display(HTML(alpha_table_8_html_wrapped))

# Save workspace (don't save connection - WRDS connections can't be pickled)
workspace = {
    'merged': merged, 'portfolio_returns': portfolio_returns,
    'portfolio_returns_8': portfolio_returns_8, 'portfolio_metrics': portfolio_metrics,
    'portfolio_metrics_8': portfolio_metrics_8, 'portfolio_returns_sector': portfolio_returns_sector,
    'market_returns': market_returns, 'sector_metrics': sector_metrics,
    'performance_table': performance_table, 'performance_table_8': performance_table_8,
    'sector_table': sector_table, 'analysis_data': analysis_data,
    'alpha_table': alpha_table, 'alpha_table_html_wrapped': alpha_table_html_wrapped,
    'start_year': start_year, 'end_year': end_year,
    'start_date': start_date, 'end_date': end_date, 'cusip_target': cusip_target
}
with open(workspace_file, 'wb') as f:
    pickle.dump(workspace, f)



Performance Table (2 portfolios):
                    Portfolio  Arithmetic Monthly Return  Annualized Return  Alpha (Monthly)      Beta  Alpha t-stat  Beta t-stat Alpha p-value Beta p-value  Alpha Lower CI (95%)  Alpha Upper CI (95%)  Beta Lower CI (95%)  Beta Upper CI (95%)  R-squared  Sharpe Ratio
0     Equal-Weighted_With_XRD                   0.012239             0.1572        -0.000037  1.294178     -0.057592       106.74     9.541e-01    0.000e+00             -0.001312              0.001237             1.270359             1.317997   0.954985      0.480351
1  Equal-Weighted_Without_XRD                   0.010946             0.1396         0.000199  1.091754      0.366730       107.51     7.140e-01    0.000e+00             -0.000868              0.001267             1.071807             1.111701   0.955607      0.494008
2     Value-Weighted_With_XRD                   0.011056             0.1410         0.000515  1.071463      0.964052        91.31     3.355e-01    0.000e+00     

Portfolio,Alpha (monthly %),t-statistic,p-value,Beta,R²,Sharpe
R&D VW,0.05,0.96,0.335,1.07,0.94,0.6
CRSP EW Benchmark,—,—,—,—,—,0.5
No R&D VW,-0.03,-0.45,0.651,0.94,0.91,0.53
R&D EW,-0.00,-0.06,0.954,1.29,0.95,0.48
No R&D EW,0.02,0.37,0.714,1.09,0.96,0.49
CRSP VW Benchmark,—,—,—,—,—,0.59


Portfolio,Alpha (monthly %),t-statistic,p-value,Beta,R²,Sharpe
R&D Investable VW,0.15*,1.84,0.066,0.95,0.85,0.63
CRSP EW Benchmark,—,—,—,—,—,0.5
R&D Not invest. VW,-0.10,-1.15,0.252,1.23,0.88,0.48
No R&D Investable VW,-0.03,-0.22,0.824,0.89,0.72,0.42
No R&D Not invest. VW,-0.03,-0.46,0.647,1.02,0.88,0.52
R&D Investable EW,0.29**,2.35,0.019,0.7,0.63,0.6
R&D Not invest. EW,-0.02,-0.34,0.735,1.32,0.95,0.47
No R&D Investable EW,0.18,1.16,0.247,0.62,0.49,0.44
No R&D Not invest. EW,0.01,0.11,0.914,1.11,0.95,0.48
CRSP VW Benchmark,—,—,—,—,—,0.59


KeyboardInterrupt: 

In [10]:
from IPython.display import HTML
HTML(alpha_table_html_wrapped)


Portfolio,Alpha (monthly %),t-statistic,p-value,Beta,R²,Sharpe
R&D VW,0.05,0.96,0.335,1.07,0.94,0.6
No R&D VW,-0.03,-0.45,0.651,0.94,0.91,0.53
R&D EW,-0.00,-0.06,0.954,1.29,0.95,0.48
No R&D EW,0.02,0.37,0.714,1.09,0.96,0.49
CRSP VW Benchmark,—,—,—,—,—,0.59
CRSP EW Benchmark,—,—,—,—,—,0.5
